# Kiểm định ý nghĩa thống kê ghép cặp (Pha U - Bước S0/E1)

Notebook này thực hiện kiểm định bootstrap ghép cặp (paired bootstrap test) và kiểm định Wilcoxon signed-rank để xác định ý nghĩa thống kê cho các so sánh sát nhau:
1. **Mô phỏng VAR tuyến tính (Pha R):** So sánh MSE ghép cặp theo cửa sổ giữa Amortized và KSG trên từng độ dài $N$.
2. **Dữ liệu chuyển mạch phi tuyến (Pha R2):** So sánh trên cấu hình Periodic Coupling.
3. **Dữ liệu tim - hô hấp thực tế (Pha U):** So sánh hiệu số TE giữa Adapted vs KSG trên các bản ghi giữ lại.

In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR / 'src') not in sys.path:
    sys.path.append(str(ROOT_DIR / 'src'))

import numpy as np
import pandas as pd
from pqrst.evaluation.significance import paired_bootstrap_test

records_table = []

# 1. So sánh ghép cặp trên dữ liệu Linear Gaussian (Pha R)
df_r = pd.read_csv(ROOT_DIR / 'results' / 'tables' / 'phase_r_grid_raw.csv')
piv_r = df_r.pivot(index=['config_name', 'n_samples', 'seed'], columns='estimator', values='error')
se_r = piv_r ** 2

print("=== 1. KIỂM ĐỊNH GHÉP CẶP MSE: AMORTIZED vs KSG (Linear Gaussian) ===")
for n_val, grp in se_r.groupby('n_samples'):
    res = paired_bootstrap_test(grp['Amortized'], grp['KSG'], n_bootstrap=1000, seed=42)
    row = {
        'experiment': 'Linear_VAR',
        'subset': f'N={n_val}',
        'comparison': 'Amortized_vs_KSG',
        'metric': 'MSE',
        'n': res['n'],
        'mean_a': res['mean_a'],
        'mean_b': res['mean_b'],
        'mean_diff': res['mean_diff'],
        'ci_low': res['ci_low'],
        'ci_high': res['ci_high'],
        'p_value_bootstrap': res['p_value_bootstrap'],
        'p_value_wilcoxon': res['p_value_wilcoxon'],
        'significant_005': res['significant_005'],
    }
    records_table.append(row)
    print(f"N={n_val:3d} | Amo={res['mean_a']:.5f} vs KSG={res['mean_b']:.5f} | diff={res['mean_diff']:+.5f} | 95% CI=[{res['ci_low']:+.5f}, {res['ci_high']:+.5f}] | p_boot={res['p_value_bootstrap']:.4f} | p_wilc={res['p_value_wilcoxon']:.2e}")

# 2. So sánh ghép cặp trên dữ liệu Periodic Coupling (Pha R2)
df_r2 = pd.read_csv(ROOT_DIR / 'results' / 'tables' / 'phase_r2_periodic_grid_raw.csv')
piv_r2 = df_r2.pivot(index=['config_name', 'n_samples', 'seed'], columns='estimator', values='error')
se_r2 = piv_r2 ** 2

print("\n=== 2. KIỂM ĐỊNH GHÉP CẶP MSE: AMORTIZED vs KSG (Periodic Coupling) ===")
for n_val, grp in se_r2.groupby('n_samples'):
    res = paired_bootstrap_test(grp['Amortized'], grp['KSG'], n_bootstrap=1000, seed=42)
    row = {
        'experiment': 'Periodic_Coupling',
        'subset': f'N={n_val}',
        'comparison': 'Amortized_vs_KSG',
        'metric': 'MSE',
        'n': res['n'],
        'mean_a': res['mean_a'],
        'mean_b': res['mean_b'],
        'mean_diff': res['mean_diff'],
        'ci_low': res['ci_low'],
        'ci_high': res['ci_high'],
        'p_value_bootstrap': res['p_value_bootstrap'],
        'p_value_wilcoxon': res['p_value_wilcoxon'],
        'significant_005': res['significant_005'],
    }
    records_table.append(row)
    print(f"N={n_val:3d} | Amo={res['mean_a']:.5f} vs KSG={res['mean_b']:.5f} | diff={res['mean_diff']:+.5f} | 95% CI=[{res['ci_low']:+.5f}, {res['ci_high']:+.5f}] | p_boot={res['p_value_bootstrap']:.4f} | p_wilc={res['p_value_wilcoxon']:.2e}")

# 3. So sánh ghép cặp trên dữ liệu thực thích nghi (Pha U)
df_u = pd.read_csv(ROOT_DIR / 'results' / 'tables' / 'phase_u_adaptation_per_record.csv')
print("\n=== 3. KIỂM ĐỊNH GHÉP CẶP HIỆU SỐ TE TRÊN DỮ LIỆU THẬT (Adapted vs KSG) ===")
res_u = paired_bootstrap_test(df_u['real_s0'], df_u['KSG'], n_bootstrap=1000, seed=42)
row_u = {
    'experiment': 'Real_Fantasia_17',
    'subset': 'Heldout_Records',
    'comparison': 'Adapted_s0_vs_KSG',
    'metric': 'TE_difference',
    'n': res_u['n'],
    'mean_a': res_u['mean_a'],
    'mean_b': res_u['mean_b'],
    'mean_diff': res_u['mean_diff'],
    'ci_low': res_u['ci_low'],
    'ci_high': res_u['ci_high'],
    'p_value_bootstrap': res_u['p_value_bootstrap'],
    'p_value_wilcoxon': res_u['p_value_wilcoxon'],
    'significant_005': res_u['significant_005'],
}
records_table.append(row_u)
print(f"Adapted={res_u['mean_a']:.4f} vs KSG={res_u['mean_b']:.4f} | diff={res_u['mean_diff']:+.4f} | 95% CI=[{res_u['ci_low']:+.4f}, {res_u['ci_high']:+.4f}] | p_boot={res_u['p_value_bootstrap']:.4f} | p_wilc={res_u['p_value_wilcoxon']:.2e}")

# Lưu bảng tổng hợp
summary_df = pd.DataFrame(records_table)
out_path = ROOT_DIR / 'results' / 'tables' / 'phase_u_significance.csv'
summary_df.to_csv(out_path, index=False)
print(f"\nĐã lưu kết quả kiểm định vào: {out_path}")
summary_df.head(15)


=== 1. KIỂM ĐỊNH GHÉP CẶP MSE: AMORTIZED vs KSG (Linear Gaussian) ===
N= 10 | Amo=0.01805 vs KSG=0.01700 | diff=+0.00106 | 95% CI=[+0.00014, +0.00194] | p_boot=0.0180 | p_wilc=8.46e-01
N= 20 | Amo=0.01122 vs KSG=0.01203 | diff=-0.00081 | 95% CI=[-0.00142, -0.00015] | p_boot=0.0180 | p_wilc=2.06e-07
N= 30 | Amo=0.00858 vs KSG=0.00917 | diff=-0.00059 | 95% CI=[-0.00104, -0.00016] | p_boot=0.0130 | p_wilc=1.63e-04
N= 50 | Amo=0.00653 vs KSG=0.00645 | diff=+0.00008 | 95% CI=[-0.00024, +0.00041] | p_boot=0.6490 | p_wilc=5.72e-01
N=100 | Amo=0.00482 vs KSG=0.00349 | diff=+0.00133 | 95% CI=[+0.00112, +0.00155] | p_boot=0.0000 | p_wilc=8.30e-19
N=200 | Amo=0.00392 vs KSG=0.00186 | diff=+0.00206 | 95% CI=[+0.00187, +0.00223] | p_boot=0.0000 | p_wilc=5.89e-72

=== 2. KIỂM ĐỊNH GHÉP CẶP MSE: AMORTIZED vs KSG (Periodic Coupling) ===
N= 10 | Amo=0.02095 vs KSG=0.02938 | diff=-0.00843 | 95% CI=[-0.01145, -0.00545] | p_boot=0.0000 | p_wilc=7.99e-07
N= 20 | Amo=0.01551 vs KSG=0.03165 | diff=-0.01614 |

,experiment,subset,comparison,metric,n,mean_a,mean_b,mean_diff,ci_low,ci_high,p_value_bootstrap,p_value_wilcoxon,significant_005
0,Linear_VAR,N=10,Amortized_vs_KSG,MSE,3000,0.018054,0.016995,0.001059,0.000143,0.001944,0.018,8.456565e-01,True
1,Linear_VAR,N=20,Amortized_vs_KSG,MSE,3000,0.011221,0.012027,-0.000807,-0.001420,-0.000152,0.018,2.064453e-07,True
2,Linear_VAR,N=30,Amortized_vs_KSG,MSE,3000,0.008582,0.009169,-0.000587,-0.001043,-0.000163,0.013,1.626352e-04,True
3,Linear_VAR,N=50,Amortized_vs_KSG,MSE,3000,0.006527,0.006445,0.000081,-0.000235,0.000406,0.649,5.717871e-01,False
4,Linear_VAR,N=100,Amortized_vs_KSG,MSE,3000,0.004823,0.003490,0.001332,0.001118,0.001547,0.000,8.295228e-19,True
5,Linear_VAR,N=200,Amortized_vs_KSG,MSE,3000,0.003917,0.001862,0.002055,0.001875,0.002231,0.000,5.890382e-72,True
6,Periodic_Coupling,N=10,Amortized_vs_KSG,MSE,1200,0.020952,0.029384,-0.008432,-0.011449,-0.005452,0.000,7.991208e-07,True
7,Periodic_Coupling,N=20,Amortized_vs_KSG,MSE,1200,0.015506,0.031648,-0.016142,-0.018609,-0.013709,0.000,1.429505e-34,True
8,Periodic_Coupling,N=30,Amortized_vs_KSG,MSE,1200,0.011773,0.032616,-0.020843,-0.023024,-0.018643,0.000,1.525912e-56,True
9,Periodic_Coupling,N=50,Amortized_vs_KSG,MSE,1200,0.009206,0.038296,-0.029090,-0.031840,-0.026528,0.000,3.552480e-74,True
